# LC 210 — Course Schedule II
**Day 42 | Graphs: Topological Sort | Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Topological order is the reverse
post-order of a DFS. Append each node to the result <em>after</em>
all its descendants are processed, then reverse. A back-edge
(GRAY node) means a cycle — return <code>[]</code>.
</div>

## Official Problem Statement

There are `numCourses` courses labelled `0` to `numCourses - 1`.
You are given an array `prerequisites` where
`prerequisites[i] = [a, b]` means you must take course `b`
**before** course `a`.

Return the ordering of courses you should take to finish all
courses. If there are many valid answers, return **any** of them.
If it is impossible to finish all courses, return an **empty**
array.

**Constraints**
- `1 <= numCourses <= 2000`
- `0 <= prerequisites.length <= 5000`
- `prerequisites[i].length == 2`
- `0 <= a, b < numCourses`
- All pairs `[a, b]` are **unique**

**Examples**
```
Input:  numCourses=2, prerequisites=[[1,0]]
Output: [0, 1]   # take 0 first, then 1

Input:  numCourses=4, prerequisites=[[1,0],[2,0],[3,1],[3,2]]
Output: [0,1,2,3] or [0,2,1,3]

Input:  numCourses=2, prerequisites=[[1,0],[0,1]]
Output: []   # cycle detected
```

## Walk Through an Example by Hand

```
numCourses=4, prerequisites=[[1,0],[2,0],[3,1],[3,2]]

Adjacency list (edge b->a):
  0: [1, 2]
  1: [3]
  2: [3]
  3: []

DFS from node 0 (post-order appending):

  dfs(0): state[0]=GRAY
    dfs(1): state[1]=GRAY
      dfs(3): state[3]=GRAY
        no neighbours
        state[3]=BLACK, result=[3]   <- append 3 first
      back in dfs(1):
      state[1]=BLACK, result=[3,1]   <- append 1
    dfs(2): state[2]=GRAY
      dfs(3): state==BLACK -> skip
      state[2]=BLACK, result=[3,1,2] <- append 2
    back in dfs(0):
    state[0]=BLACK, result=[3,1,2,0] <- append 0

result[::-1] = [0, 2, 1, 3]   <- valid topological order
```

## What This Is Actually Asking

We need to produce a linear ordering of courses such that every
prerequisite appears before the course that depends on it.
This is the classical **topological sort** problem on a directed
graph, and it has a solution only when the graph is a DAG
(no cycles).
The reverse post-order DFS collects each node only after all its
downstream dependencies are resolved, naturally producing a valid
topological sequence when reversed.

## The Picture

```
Graph: 0 -> 1 -> 3
       0 -> 2 -> 3

DFS state legend:
  0 = WHITE  (unvisited)
  1 = GRAY   (on current DFS stack — visiting)
  2 = BLACK  (fully explored — done)

Post-order collection:
  A node is appended to `result` AFTER all its neighbours
  finish (when we colour it BLACK).

  Step       node   state[]       result (post-order)
  -------    ----   ----------    -------------------
  enter        0    [1,0,0,0]     []
  enter        1    [1,1,0,0]     []
  enter        3    [1,1,0,1]     []
  exit         3    [1,1,0,2]     [3]          <- leaf first
  exit         1    [1,2,0,2]     [3, 1]
  enter        2    [1,2,1,2]     [3, 1]
    nbr 3 = BLACK -> skip
  exit         2    [1,2,2,2]     [3, 1, 2]
  exit         0    [2,2,2,2]     [3, 1, 2, 0] <- root last

  reverse ->  [0, 2, 1, 3]   valid topological order

Cycle detection:
  If neighbour is GRAY (1) -> back-edge -> return [] immediately
```

## When To Use This Pattern

- When you need an ordered list of tasks respecting dependencies,
  think **topological sort (reverse post-order DFS)**.
- When the problem could have multiple valid orderings and any
  valid one is acceptable, think **DFS post-order + reverse**.
- When you need to both detect cycles AND produce an order,
  think **three-colour DFS** (not just visited/unvisited).
- When a dependency graph might be disconnected, think
  **outer loop over all nodes** to ensure full coverage.
- When the graph is guaranteed acyclic (DAG), think
  **Kahn's BFS** as a clean alternative to DFS.

## The Approach

Build an adjacency list (edge b->a for each [a,b] prerequisite).
Maintain a `state` array (0/1/2) and an empty `result` list.
Run DFS from every unvisited node: mark GRAY on entry, recurse
into all neighbours (return `[]` on a GRAY hit), then mark BLACK
and **append the current node** to `result` on exit.
After all DFS calls complete, return `result` reversed.

In [ ]:
from typing import List
from collections import defaultdict, deque

In [ ]:
def test_harness(func):
    """Run test cases for LC 210 findOrder."""

    def is_valid_order(order, n, prereqs):
        """Check each prereq [a,b]: b appears before a."""
        if len(order) != n:
            return False
        pos = {course: i for i, course in enumerate(order)}
        return all(pos[b] < pos[a] for a, b in prereqs)

    cases = [
        (2, [[1, 0]], True),
        (2, [[1, 0], [0, 1]], False),
        (1, [], True),
        (4, [[1,0],[2,0],[3,1],[3,2]], True),
        (4, [[1,0],[2,1],[3,2],[1,3]], False),
        (3, [[0,1],[0,2],[1,2]], True),
    ]
    passed = 0
    for i, (n, prereqs, expect_valid) in enumerate(cases):
        result = func(n, prereqs)
        if expect_valid:
            ok = is_valid_order(result, n, prereqs)
        else:
            ok = (result == [])
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"n={n} -> {result}"
        )
    print(f"\nResult: {passed}/{len(cases)} passed")

In [ ]:
def findOrder(
    numCourses: int,
    prerequisites: List[List[int]]
) -> List[int]:
    """
    Return a valid topological ordering of courses.

    Uses DFS reverse post-order with three states:
      0 = unvisited (WHITE)
      1 = visiting  (GRAY)  — on current DFS stack
      2 = done      (BLACK) — fully explored

    Each node is appended to `result` after all its
    descendants finish (post-order). Reversing `result`
    gives a valid topological order.

    If a GRAY neighbour is encountered a cycle exists
    and [] is returned.

    Args:
        numCourses:    total number of courses
        prerequisites: [a, b] pairs meaning b before a

    Returns:
        List[int]: valid order, or [] if cycle detected

    Time:  O(V + E)
    Space: O(V + E)
    """
    print(f"[DEBUG] numCourses={numCourses}, "
          f"prerequisites={prerequisites}")

    graph = defaultdict(list)
    for a, b in prerequisites:
        graph[b].append(a)  # edge b -> a

    print(f"[DEBUG] graph={dict(graph)}")

    state = [0] * numCourses
    result = []

    def dfs(node: int) -> bool:
        """
        Return True if cycle detected.
        Append node to result in post-order.
        """
        if state[node] == 1:  # GRAY -> cycle
            print(f"[DEBUG] cycle at node {node}")
            return True
        if state[node] == 2:  # BLACK -> safe
            return False

        state[node] = 1       # mark GRAY
        for neighbour in graph[node]:
            if dfs(neighbour):
                return True
        state[node] = 2       # mark BLACK
        result.append(node)   # post-order append
        print(
            f"[DEBUG] appended {node}, "
            f"result so far={result}"
        )
        return False

    for course in range(numCourses):
        if state[course] == 0:
            if dfs(course):
                return []

    pass  # replace with: return result[::-1]

In [ ]:
# Uncomment and run when solution is ready
# test_harness(findOrder)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (all permutations) | O(V!) | O(V) | Infeasible |
| DFS reverse post-order (optimal) | O(V+E) | O(V+E) | This solution |
| Kahn's BFS (in-degree) | O(V+E) | O(V+E) | Clean alt. |

V = numCourses, E = len(prerequisites)

## Real World Connection

In **AWS Glue** and **Apache Airflow**, every pipeline is a DAG
where each job must run only after its upstream dependencies
complete. The scheduler internally performs a topological sort to
derive the execution order before submitting any work. At **Citi**,
regulatory reporting frameworks orchestrate hundreds of data
transformation steps with strict ordering constraints — an
incorrect order can produce wrong P&L figures sent to regulators.
For **Data Engineers**, understanding topological sort is
essential for building reliable dbt model chains, debugging
circular references in Spark lineage graphs, and reasoning about
any dependency-driven workflow.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra